In [1]:
import os
from pathlib import Path
import json

from src.run_whisper_bls import transcribe_with_optional_bias

# beam hook関数は既存のBeam Seachの改良：
# ビームの結果だけでなく，途中の候補を意図的に参照し，目的のtokenと合致したらその累積確率を強制的にmaxにして以降の探索で参照させる役割
# 詳しくはTeamsの2段組論文の音声パートを見ると分かる
from src.beam_hook import InspectConfig

"""# 1. プロジェクトのルートディレクトリ
BASE_DIR = "/home/medical/WorkSpace/MedWhisper_nachi"
os.chdir(BASE_DIR)
# 2. 音声データフォルダのパス
DATA_DIR = os.path.join(BASE_DIR, "2025115cleandata", "スマホ")"""

PROJECT_ROOT = Path.cwd()
"実施者のみ"
WAV = str(PROJECT_ROOT / "data" / "2025115cleandata" / "スマホ" / "1_川村先生_練習.m4a")
"実施者，協力者"
#WAV = "/root/MedWhisper/2025115cleandata/スマホ/3_川村先生_協力者_大野.m4a"
"実施者，協力者，AED"
WAV = str(PROJECT_ROOT / "data" / "2025115cleandata" / "スマホ" / "4_川村先生_協力者_大場AED.m4a")
#WAV ="/root/MedWhisper/20241018/右後_2回目_川村先生.wav"
OUT = str(PROJECT_ROOT / "outputs" / "transcription" / "out_whisper2")
Path(OUT).mkdir(parents=True, exist_ok=True)


# 読み込もうとしている音声ファイルのパス変数（例: audio_path や file_path など）
print("確認するパス:", WAV)
print("ファイルは実在するか:", os.path.exists(WAV))
print("読み取り権限はあるか:", os.access(WAV, os.R_OK))

DOMAIN_TERMS = [
    # "傷","傷病者"
]


inspect_cfg = InspectConfig(
    enable_inspect=True,
    # beam の途中候補でtopいくつを残すか選択できる
    topk=5,
    # これらで指定した語句と途中の候補が合えば，強制的に確率を高める
    targets=["傷","周囲","体","胸"],
   # targets = []
   # どのくらい強制力を持たせるか
    force_bonus=50,
    lock_after_hit=True,
)

# inspect_cfg.ban_token_ids = [15553] この値は文字化けしたトークン　
  
r1 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    #initial_prompt="傷 体 胸 ",  # これは既存のイニシャルプロンプトです
    #use_bias=True,
    inspect_cfg=inspect_cfg, #提案手法を使わない場合，こちらをコメントアウトしてください
)

print(r1.get("text",""))

確認するパス: /media/dl-box/ADATA SE800/med/MedWhisper_nachi/2025115cleandata/スマホ/4_川村先生_協力者_大場AED.m4a
ファイルは実在するか: True
読み取り権限はあるか: True


/media/dl-box/ADATA SE800/med/MedWhisper_nachi/env/lib/python3.8/site-packages/torch/cuda/__init__.py:128: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/media/dl-box/ADATA SE800/med/MedWhisper_nachi/env/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


M25000 河村雄貴 これからBLSを開始します傷病者発見周囲は安全です 感染防御に配慮します大丈夫ですか 大丈夫ですか 大丈夫ですか誰か誰か誰か来てくださいあなたは119番通報してくださいあなた、AEDを持ってきてください。必ずここに戻ってきてください。胸とお腹を見て、呼吸の確認。同時に脈の確認。呼吸、脈ありません。胸骨圧迫を開始します。1、2、3、4、5、6、7、8、9、102、2、3、4、5、6、7、8、9、103、2、3、4、5、6、7、8、9、10AED持ってきましたあなたAED使えますか?使えません胸骨圧迫を変わってください1、2の3で変わりましょうせーの、1、2、31、2、3、4、5、6、7、8、9、10、2、2、3、4、5、6、7、8、9、10、2、2、3、4、5、6、7、8、9体表面よし体に触れないでください離れてください安全確認します私は離れてますあなたは離れてますみんな離れてます体から離れてください離れてください私はあなたは離れてますみんな離れてますショックします胸骨圧迫を開始します1、2、3、4、5、6、7、8、9、10、2、2、3救急体です救急体の方この方3分前に目の前で倒れるところを見ました胸骨圧迫をしてAEDで1体ショックをしています意識はまだ戻っていませんこの人の身元はわからないですがこの人の荷物はこの足元にありますので一緒に持っていってください引き継ぎます以上


In [2]:
import re
from collections import Counter

def count_terms(text, terms):
    return {t: len(re.findall(re.escape(t), text)) for t in terms}

def basic_stats(text):
    return {
        "len": len(text),
        "chars": Counter(text).most_common(5),
    }

# 1) no-bias
r0 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    use_bias=False,
)

# 2) with-bias
r1 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    use_bias=True,
    inspect_cfg=inspect_cfg,
)

t0 = r0.get("text","")
t1 = r1.get("text","")

print("=== NO BIAS ===")
print(t0)
print("terms:", count_terms(t0, DOMAIN_TERMS))
print("stats:", basic_stats(t0))

print("\n=== WITH BIAS ===")
print(t1)
print("terms:", count_terms(t1, DOMAIN_TERMS))
print("stats:", basic_stats(t1))

print("\n=== DIFF ===")
print("same_text:", t0 == t1)
print("delta_len:", len(t1) - len(t0))
for term in DOMAIN_TERMS:
    print(term, count_terms(t1,[term])[term] - count_terms(t0,[term])[term])

/media/dl-box/ADATA SE800/med/MedWhisper_nachi/env/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/media/dl-box/ADATA SE800/med/MedWhisper_nachi/env/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== NO BIAS ===
M25000 河村雄貴 これからBLSを開始します症病者発見 周囲は安全です 感染防御に配慮します大丈夫ですか? 大丈夫ですか? 大丈夫ですか?誰か誰か誰か来てくださいあなたは119番通報してくださいあなた、AEDを持ってきてください。必ずここに戻ってきてください。胸とお腹を見て、呼吸の確認。同時に脈の確認。呼吸、脈ありません。胸骨圧迫を開始します。1、2、3、4、5、6、7、8、9、102、2、3、4、5、6、7、8、9、103、2、3、4、5、6、7、8、9、10AED持ってきましたあなたAED使えますか?使えませんじゃあ胸骨圧迫を変わってください1、2の3で変わりましょうせーの、1、2、31、2、3、4、5、6、7、8、9、10、2、2、3、4、5、6、7、8、9、10、2、2、3、4、5、6、7、8、9対応面、よし心電図を解析中です体に振れないでください離れてください安全確認します私は離れてますあなたは離れてますみんな離れてます離れてください私はあなたは離れてますみんな離れてますショックしますショックが完了しました一時中断胸骨圧迫を開始します1、2、3、4、5、6、7、8、9、10、2、2、3救急隊です救急隊の方この方3分前に目の前で倒れるところを見ました胸骨圧迫をしてAEDで1回ショックをしています意識はまだ戻っていませんこの人の身元はわからないですがこの人の荷物はこの足元にありますので一緒に持っていってください引き継ぎます以上
terms: {}
stats: {'len': 629, 'chars': [('、', 75), ('ま', 23), ('す', 22), ('て', 21), ('2', 15)]}

=== WITH BIAS ===
M25000 河村雄貴 これからBLSを開始します傷病者発見周囲は安全です 感染防御に配慮します大丈夫ですか 大丈夫ですか 大丈夫ですか誰か誰か誰か来てくださいあなたは119番通報してくださいあなた、AEDを持ってきてください。必ずここに戻ってきてください。胸とお腹を見て、呼吸の確認。同時に脈の確認。呼吸、脈ありません。胸骨圧迫を開始します。1、2、3、4、5、6、7、8、9、102、2、3、4、5、6、7、8、9、103、2、3、4、5、6、7、8、9、10AED持ってき

In [3]:
from whisper.tokenizer import get_tokenizer
tok = get_tokenizer(multilingual=True, task="transcribe")

print("timestamp_begin:", tok.timestamp_begin)
print("eot:", tok.eot)
print("15553 is timestamp?",
      tok.timestamp_begin <= 15553 < tok.eot)

timestamp_begin: 50364
eot: 50257
15553 is timestamp? False


In [4]:
from whisper.tokenizer import get_tokenizer
tok = get_tokenizer(multilingual=True, task="transcribe")
print(tok.decode([15553]))

�


In [5]:

p = Path(OUT)
cand = sorted([x for x in p.glob("**/*") if x.is_file() and x.suffix in (".jsonl",".json")])
print("\n=== OUT DIR artifacts ===")
for x in cand:
    print(str(x))

# 下記ファイルにてにてbeam探索過程の途中のtoken候補が見れる
inspect_path = Path(OUT) / "inspect.jsonl"
beamlog_path = Path(OUT) / "result_bias_beamlog.jsonl"  

print("\n=== Expected ===")
print("inspect:", inspect_path, "exists=", inspect_path.exists())
print("beamlog:", beamlog_path, "exists=", beamlog_path.exists())


WATCH_TARGETS = ["傷", "傷病者", "傷病者発見", " 発見", "傷病"]  # 必要に応じて追加

def summarize_inspect_hits(inspect_jsonl: Path, watch_targets=None, phase="pre_bias", max_show=50):
    if watch_targets is None:
        watch_targets = []
    if not inspect_jsonl.exists():
        print(f"[warn] not found: {inspect_jsonl}")
        return

    total = 0
    pre_lines = 0
    hit_rows = []

    with inspect_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                rec = json.loads(line)
            except Exception:
                continue

            if rec.get("phase") == phase:
                pre_lines += 1

                if rec.get("hits_best"):
                    hb = rec["hits_best"]
                    # watch_targets に含まれる target が一つでもあれば hit扱い
                    keep = False
                    for it in hb:
                        t = it.get("target","")
                        if (not watch_targets) or (t in watch_targets):
                            keep = True
                    if keep:
                        hit_rows.append(rec)
                else:
            
                    hits = rec.get("hits") or []
                    keep = False
                    for it in hits:
                        t = it.get("target","")
                        if (not watch_targets) or (t in watch_targets):
                            keep = True
                    if keep:
                        hit_rows.append(rec)

    print(f"[info] lines={total}, {phase}_lines={pre_lines}")

    # steps->beams をまとめる
    step2beams = {}
    for rec in hit_rows:
        step = rec.get("step")
        beam = rec.get("beam")
        if step is None or beam is None:
            continue
        step2beams.setdefault(step, set()).add(beam)

    print("\n=== steps where WATCH appeared in hits ===")
    for step in sorted(step2beams.keys()):
        beams = sorted(step2beams[step])
        print(f"step={step}: beams={beams}")

    print("\n=== details (only where WATCH appeared) ===")
    shown = 0
    for rec in hit_rows:
        if shown >= max_show:
            break
        step = rec.get("step")
        beam = rec.get("beam")
        targets = rec.get("targets")

        if rec.get("hits_best"):
            for it in rec["hits_best"]:
                print(
                    f"step={step} beam={beam} target={it.get('target')} "
                    f"rank={it.get('rank')} lp={it.get('logprob')} gap={it.get('gap_to_top1')} "
                    f"decoded={it.get('decoded')}"
                )
        else:

            hits = rec.get("hits") or []
            if hits:
                it = hits[0]
                print(
                    f"step={step} beam={beam} watch_logprob={it.get('logprob')} "
                    f"target={it.get('target')} decoded={it.get('decoded')} targets={targets}"
                )
        shown += 1

    
    print("\n=== target hit summary (rough) ===")
    counts = {t: {"seen": 0, "hit": 0} for t in (watch_targets or [])}
    if watch_targets:
        with inspect_jsonl.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                if rec.get("phase") != phase:
                    continue
                for t in watch_targets:
                    counts[t]["seen"] += 1
                # 改修版
                if rec.get("hits_best"):
                    hit_targets = set([it.get("target","") for it in rec["hits_best"]])
                    for t in watch_targets:
                        if t in hit_targets:
                            counts[t]["hit"] += 1
                else:
          
                    hit_targets = set([it.get("target","") for it in (rec.get("hits") or [])])
                    for t in watch_targets:
                        if t in hit_targets:
                            counts[t]["hit"] += 1

        for t, d in counts.items():
            print(f"{t}: seen={d['seen']}, hit={d['hit']}")


summarize_inspect_hits(inspect_path, watch_targets=WATCH_TARGETS, phase="pre_bias", max_show=50)


summarize_inspect_hits(inspect_path, watch_targets=WATCH_TARGETS, phase="post_bias", max_show=50)


=== OUT DIR artifacts ===
/media/dl-box/ADATA SE800/med/MedWhisper_nachi/out_whisper2/inspect.jsonl
/media/dl-box/ADATA SE800/med/MedWhisper_nachi/out_whisper2/result_bias_beamlog.jsonl
/media/dl-box/ADATA SE800/med/MedWhisper_nachi/out_whisper2/result_default_beamlog.jsonl

=== Expected ===
inspect: /media/dl-box/ADATA SE800/med/MedWhisper_nachi/out_whisper2/inspect.jsonl exists= True
beamlog: /media/dl-box/ADATA SE800/med/MedWhisper_nachi/out_whisper2/result_bias_beamlog.jsonl exists= True
[info] lines=4182, pre_bias_lines=2091

=== steps where WATCH appeared in hits ===
step=20: beams=[2]
step=21: beams=[0, 1]
step=27: beams=[2]
step=28: beams=[0, 1]
step=36: beams=[2]
step=37: beams=[0, 1]
step=40: beams=[2]
step=41: beams=[0, 1]

=== details (only where WATCH appeared) ===
step=20 beam=2 target=傷 rank=2 lp=13.556885719299316 gap=2.536715507507324 decoded=傷
step=21 beam=0 target=傷 rank=2 lp=13.593132019042969 gap=2.5696277618408203 decoded=傷
step=21 beam=1 target=傷 rank=2 lp=13.79